CREATE DATABASE ON SQLLITE

In [ ]:
import sqlite3

conn = sqlite3.connect("smart_parking.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS parking_events (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    slot_id TEXT,
    distance_cm REAL,
    status TEXT,
    ts INTEGER
)
""")

conn.commit()
conn.close()

print("Database and table created successfully")

Database and table created successfully


In [ ]:
!pip install paho-mqtt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 2.6 MB/s eta 0:00:00


RECEIVE DATA FROM HIVEMQ & STORE IT IN SQL


In [ ]:
import ssl
import json
import sqlite3
import paho.mqtt.client as mqtt

BROKER = "26a2fce6355d4b8c82ea0a348575f01d.s1.eu.hivemq.cloud"
PORT = 8883
USERNAME = "parkinguser"
PASSWORD =  "Parking123"
TOPIC = "parking/#"
DB_PATH = "smart_parking.db"

def save_to_db(data):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO parking_events (slot_id, distance_cm, status, ts)
        VALUES (?, ?, ?, ?)
    """, (
        data.get("slot_id", "S1"),
        float(data.get("distance_cm", -1)),
        data.get("status", "unknown"),
        int(data.get("ts", 0))
    ))

    conn.commit()
    conn.close()

def on_connect(client, userdata, flags, rc):
    print("Connected with result code:", rc)
    client.subscribe(TOPIC)
    print("Subscribed to:", TOPIC)

def on_message(client, userdata, msg):
    try:
        payload = msg.payload.decode("utf-8")
        data = json.loads(payload)
        save_to_db(data)
        print("Stored in SQLite:", data)
    except Exception as e:
        print("Error:", e)

client = mqtt.Client()
client.username_pw_set(USERNAME, PASSWORD)

client.tls_set(cert_reqs=ssl.CERT_REQUIRED, tls_version=ssl.PROTOCOL_TLS)
client.tls_insecure_set(False)

client.on_connect = on_connect
client.on_message = on_message

client.connect(BROKER, PORT, 60)
client.loop_start()
print("MQTT listener started in background")

/tmp/ipykernel_22159/2769905800.py:44: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client()


MQTT listener started in background


In [ ]:
import time
import sqlite3
import pandas as pd
from IPython.display import clear_output, display

for _ in range(20):   # 20 refreshes
    conn = sqlite3.connect("smart_parking.db")
    df = pd.read_sql_query("""
    SELECT * FROM parking_events
    ORDER BY id DESC
    LIMIT 5
    """, conn)
    conn.close()

    clear_output(wait=True)
    display(df)
    time.sleep(5)

,id,slot_id,distance_cm,status,ts


CHECK DATABASE RECORD

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("smart_parking.db")
df = pd.read_sql_query("SELECT * FROM parking_events", conn)
conn.close()

df

,id,slot_id,distance_cm,status,ts


BASIC QUERY (Total occupied vs free)

In [ ]:
conn = sqlite3.connect("smart_parking.db")
df_status = pd.read_sql_query("""
SELECT status, COUNT(*) as count
FROM parking_events
GROUP BY status
""", conn)
conn.close()

df_status

Stored in SQLite: {'slot_id': 'S1', 'distance_cm': 42.65205, 'status': 'free', 'ts': 4061}
Stored in SQLite: {'slot_id': 'S1', 'distance_cm': 42.65205, 'status': 'free', 'ts': 4061}
Stored in SQLite: {'slot_id': 'S1', 'distance_cm': 42.65205, 'status': 'free', 'ts': 4061}


,status,count
0,free,123
1,occupied,94


Stored in SQLite: {'slot_id': 'S1', 'distance_cm': 42.65205, 'status': 'free', 'ts': 4061}


LATEST PARKING STATUS

In [ ]:
conn = sqlite3.connect("smart_parking.db")
latest = pd.read_sql_query("""
SELECT * FROM parking_events
ORDER BY id DESC
LIMIT 5
""", conn)
conn.close()

latest

,id,slot_id,distance_cm,status,ts
0,219,S1,42.65205,free,4061
1,218,S1,42.65205,free,4061
2,217,S1,42.65205,free,4061
3,216,S1,42.65205,free,4061
4,215,S1,42.65205,free,4061


In [ ]:
from google.colab import files
files.download("smart_parking.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DB_PATH = "/content/drive/MyDrive/smart_parking.db"

In [ ]:
!pip install plotly pandas ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 35.9 MB/s eta 0:00:00


In [ ]:
import sqlite3
import pandas as pd
import plotly.express as px
from IPython.display import display, HTML

DB_PATH = "smart_parking.db"

# ---- Settings ----
TOTAL_SLOTS = 4   # change if needed
MQTT_STATUS = "Connected"   # manually change to Connected / Disconnected for now
# ------------------

def load_data():
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("SELECT * FROM parking_events ORDER BY id ASC", conn)
    conn.close()
    return df

def get_latest_status_per_slot(df):
    latest_slots = df.sort_values("id").groupby("slot_id").tail(1)
    return latest_slots

def show_dashboard():
    df = load_data()

    if df.empty:
        print("No data found in database.")
        return

    latest = df.iloc[-1]

    total_records = len(df)
    occupied_count = (df["status"] == "occupied").sum()
    free_count = (df["status"] == "free").sum()
    occupancy_rate = round((occupied_count / total_records) * 100, 2) if total_records > 0 else 0

    status_color = "#ff4b4b" if latest["status"] == "occupied" else "#2ecc71"
    mqtt_color = "#2ecc71" if MQTT_STATUS.lower() == "connected" else "#ff4b4b"

    # latest status per slot
    latest_slots = get_latest_status_per_slot(df)
    occupied_slots_now = (latest_slots["status"] == "occupied").sum()

    # parking full check
    parking_full = occupied_slots_now >= TOTAL_SLOTS

    # -------- Top cards --------
    display(HTML(f"""
    <div style="display:flex; gap:20px; flex-wrap:wrap; font-family:Arial;">
        <div style="padding:18px; border-radius:14px; background:#f8f9fa; border:1px solid #ddd; min-width:240px;">
            <h3 style="margin:0 0 10px 0;">Latest Slot Status</h3>
            <p><b>Slot:</b> {latest['slot_id']}</p>
            <p><b>Distance:</b> {round(latest['distance_cm'],2)} cm</p>
            <p><b>Status:</b> <span style="color:{status_color}; font-weight:bold;">{str(latest['status']).upper()}</span></p>
            <p><b>Timestamp:</b> {latest['ts']}</p>
        </div>

        <div style="padding:18px; border-radius:14px; background:#f8f9fa; border:1px solid #ddd; min-width:220px;">
            <h3 style="margin:0 0 10px 0;">Summary</h3>
            <p><b>Total Records:</b> {total_records}</p>
            <p><b>Occupied Count:</b> {occupied_count}</p>
            <p><b>Free Count:</b> {free_count}</p>
            <p><b>Occupancy Rate:</b> {occupancy_rate}%</p>
        </div>

        <div style="padding:18px; border-radius:14px; background:#f8f9fa; border:1px solid #ddd; min-width:220px;">
            <h3 style="margin:0 0 10px 0;">MQTT Connection</h3>
            <p><b>Status:</b> <span style="color:{mqtt_color}; font-weight:bold;">{MQTT_STATUS}</span></p>
        </div>
    </div>
    """))

    # -------- Parking Full Alert --------
    if parking_full:
        display(HTML("""
        <div style="
            margin-top:15px;
            padding:16px;
            border-radius:12px;
            background:#ffdddd;
            border:1px solid #ff4b4b;
            color:#a40000;
            font-family:Arial;
            font-size:18px;
            font-weight:bold;">
            ⚠ Parking Full Alert: All slots are occupied.
        </div>
        """))
    else:
        display(HTML(f"""
        <div style="
            margin-top:15px;
            padding:16px;
            border-radius:12px;
            background:#ddffdd;
            border:1px solid #2ecc71;
            color:#006400;
            font-family:Arial;
            font-size:18px;
            font-weight:bold;">
            ✅ Parking Available: {TOTAL_SLOTS - occupied_slots_now} slot(s) free.
        </div>
        """))

    # -------- Slot Map Layout --------
    slot_boxes = ""
    # create default slots if not present
    current_status = {f"S{i+1}": "free" for i in range(TOTAL_SLOTS)}
    for _, row in latest_slots.iterrows():
        current_status[str(row["slot_id"])] = str(row["status"]).lower()

    for slot, status in current_status.items():
        color = "#ff4b4b" if status == "occupied" else "#2ecc71"
        text = "OCCUPIED" if status == "occupied" else "FREE"
        slot_boxes += f"""
        <div style="
            width:140px;
            height:100px;
            border-radius:12px;
            margin:10px;
            display:flex;
            flex-direction:column;
            justify-content:center;
            align-items:center;
            background:{color};
            color:white;
            font-family:Arial;
            font-weight:bold;
            box-shadow:0 2px 6px rgba(0,0,0,0.2);">
            <div style="font-size:22px;">{slot}</div>
            <div style="font-size:16px;">{text}</div>
        </div>
        """

    display(HTML(f"""
    <div style="margin-top:20px; font-family:Arial;">
        <h3>Slot Map Layout</h3>
        <div style="display:flex; flex-wrap:wrap;">
            {slot_boxes}
        </div>
    </div>
    """))

    # -------- Status count chart --------
    status_count = df["status"].value_counts().reset_index()
    status_count.columns = ["status", "count"]

    fig1 = px.bar(
        status_count,
        x="status",
        y="count",
        color="status",
        title="Occupied vs Free Count",
        text="count"
    )
    fig1.update_layout(xaxis_title="Status", yaxis_title="Count")
    fig1.show()

    # -------- Distance trend --------
    fig2 = px.line(
        df,
        x="id",
        y="distance_cm",
        color="status",
        markers=True,
        title="Distance Trend Over Time"
    )
    fig2.update_layout(xaxis_title="Record ID", yaxis_title="Distance (cm)")
    fig2.show()

    # -------- Recent history --------
    print("Recent Parking Events")
    display(df.sort_values("id", ascending=False).head(10))

show_dashboard()

Recent Parking Events


,id,slot_id,distance_cm,status,ts
3,4,4,50.2,free,None
2,3,3,12.3,occupied,None
1,2,2,45.0,free,None
0,1,1,10.5,occupied,None


Auto Refresh Version

In [ ]:
import time
from IPython.display import clear_output

for i in range(20):   # 20 refresh cycles
    clear_output(wait=True)
    show_dashboard()
    time.sleep(5)     # every 5 seconds

Recent Parking Events


,id,slot_id,distance_cm,status,ts
3,4,4,50.2,free,None
2,3,3,12.3,occupied,None
1,2,2,45.0,free,None
0,1,1,10.5,occupied,None
